# ALSM v3: Hardware Measurement Notebook
## Active Learning Sparse Measurement — Single-Electron Pump Plateau Characterization

**Purpose**: Real pump measurement via GPIB hardware.  
**For simulation / algorithm testing** → use `sim_ALSM_kNN.ipynb`

---

**Method**: Schoinas 2024 (Appl. Phys. Lett. 125, 124001)  
**Convention**: $\eta = \log_{10}(|\langle n \rangle - 1|)$ — more negative = better quantization

**Hardware** (skill §2.1):
- GPIB43::8 = $V_\text{exit}$ (exit gate, Yokogawa 7651)
- GPIB43::1 = $V_\text{ent}$ (entrance gate, Yokogawa 7651)
- GPIB43::19 = DMM (Keithley 2000 via Ithaco preamp)

**Outputs** (reproducing paper Fig.1):
- (c) Variance heatmap at intermediate AL iteration + selected points
- (d) Sparse measured $\eta_M$ points after all AL iterations
- (e) Interpolated $\eta_M$ map with $\eta_\text{noise}$ contour
- (f) Schoinas nonlinear $\eta_E$ fit at fixed $V_\text{ent}$
- (g) 2D extrapolated $\eta_E(V_\text{exit}, V_\text{ent})$ map

---

## 변경 이력 (Changelog)

| 버전 | 날짜 | 변경 내용 |
|------|------|-----------|
| **v3_20260427** | 2026-04-27 | **Single source of truth for config**: cell[3] `ALSMConfig` 가 모든 실험 파라미터(V 범위, sim_mode, f_drive, amp_gain 포함)의 단일 편집 지점이 됨. cell[20] 실행 셀에서 override 4줄 제거 — 인스턴스 생성·연결·실행만 담당. v2의 "class default + run-cell override" 이중 구조가 archive 노트북 사후 분석 시 혼선(어느 값이 실제 사용됐는지 불명)을 유발해서 수정. `ALSMConfig` 내부도 [매 실험] / [드물게] / [고정] 3 섹션으로 그룹화하고 각 필드 수정 빈도를 docstring에 명시. |\n| **v2_20260420-b** | 2026-04-20 | **(a) 데이터 소스 카운트 표시·저장**: `driver_alsm()` 종료 시 `n_measured`(실측) / `n_interpolated`(griddata linear, convex hull 내부) / `n_extrapolated`(k-NN fallback, convex hull 외부) / `n_total_grid` 및 소요시간(초/분)을 콘솔 출력하고 `results.npz`에 함께 저장. **(b) `config_parmeter` 저장**: 하드웨어 설정(GPIB 주소·amp_gain·MAX_STEP_V·세틀 타임 등) + ALSM 설정(`ALSMConfig` 모든 public 스칼라 필드) + 런 메타데이터(timestamp, output_dir, 카운트, 소요시간)를 dict로 병합해 `results.npz` 안에 `config_parmeter` 키로 저장(사용자 요청 철자 보존). **(c) 코드 흐름도**: `ALSM_v2_flowchart.md` 신규 작성, 매뉴얼(`ALSM_v2_manual.md`)에 흐름도 섹션 추가. **(d) `MEAS_SETTLE` 300→100 ms**: Ithaco 1 nA/V (BW ~30 Hz, τ≈5 ms) 기준 100 ms = 22τ settling으로 충분. 1600점당 ~320 s 단축. 스킬 §8 권장 범위 하한 준수. 다른 gain에서는 매뉴얼 §12 참조. **(e) Fig.1(g) 라벨 명확화 (Proposal A+C)**: 왼쪽 2D 맵에 `V_opt(V_ent)` 궤적(cyan) overlay — 오른쪽 패널 η_E^min이 '어느 V_exit에서 읽힌 값인지' 시각적으로 연결. 오른쪽 패널 ylabel을 `η_E^min (at self-optimal V_exit)`로, 제목·annotation에 `V_opt`를 명시하여 '고정된 V_exit 값'이라는 오해 제거. **(f) 스킬 보강**: `lab-pump-measurement` 스킬 §3·§7·§12에 오늘 확정된 규약(ramp 외 직접 write 금지 / OD 쿼리로 추적 변수 동기화 / scan history 로그 필수 / connect를 try 블록에 / close에서 OD 재쿼리 / UTF-8 로그) 및 Past Failures F10–F14 추가. |
| **v2_20260420** | 2026-04-20 | **(a)** 전압 스캔 히스토리 로그 추가 (`log_scanhistory_YYYYMMDD_HHMMSS.txt`, UTF-8): `set_voltages()` 내 매 4 mV 스텝을 기록, abort 시에도 보존 (즉시 flush). skill Rule No.1 크로스체크용. **(b)** `connect()` 4 mV 위반 버그 수정: 이전 버전의 `S0.000000E` 직접 쓰기는 이전 실행이 비정상 종료된 경우 단일 명령으로 >4 mV 점프를 유발할 수 있었음. 수정: OD 쿼리로 실제 전압을 읽어 추적 변수 동기화 후, `set_voltages(0,0)`로 안전하게 0 V 이동. **(c) Opus 재검토 수정**: OD 응답 파싱 regex를 `+.5000E+00`(leading zero 없는 form)도 허용; Cell 20에서 `connect()`를 try 블록 안으로 이동하여 연결 중 실패 시에도 `close()` 실행 보장; `close()`에 OD 재쿼리 추가 (부분 연결 실패 시 추적 변수 최신화). |
| v2_20260416 | 2026-04-18 | `_ST` 딕셔너리 플롯 스타일(Tier 2) 적용; Cell 20 하드웨어 셀 주석 해제; `f_drive` 130 MHz 수정; `connect()`에 `S0.000000E` 직접 쓰기 추가 ⚠️ **(v20260420에서 4 mV 위반 원인으로 판명되어 철회)**. |
| v2_20260413 | 2026-04-13 | 최초 하드웨어 버전 (ALSM_combined 에서 분리). |


In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.interpolate import griddata
from scipy.optimize import curve_fit           # Schoinas η_E nonlinear fit (skill §2)
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import time
import os
import shutil
import re
from datetime import datetime

try:
    import pyvisa
    PYVISA_AVAILABLE = True
except ImportError:
    PYVISA_AVAILABLE = False
    print("[WARN] pyvisa not available — hardware mode disabled")

## Configuration

In [ ]:
class ALSMConfig:
    """All parameters for an ALSM run.

    실험자 단일 편집 지점 (v3): 매 실험마다 이 클래스에서 직접 수정.
    cell[20] 실행 셀은 인스턴스 생성·실행만 담당 (override 없음).

    수정 빈도 가이드:
      [매 실험] V_exit_min/max, V_ent_min/max, sim_mode, f_drive, amp_gain
      [드물게]  N_coarse, N_meas, N_AL, k_neighbors, eta_noise, eta_max
      [고정]    e_charge, n_exit, n_ent, I_sat_nA
    """

    # ===== [매 실험 수정] 측정 조건 =========================================

    # --- Mode ---
    sim_mode = False      # False: 하드웨어,  True: sim_func 사용
    sim_func = None       # sim_mode=True 일 때 측정 함수

    # --- Gate voltage ranges (소자/plateau 위치에 맞게 조정) ---
    V_exit_min = -0.5     # V_exit (exit gate) range
    V_exit_max = -0.22
    V_ent_min  = -0.5     # V_ent (entrance gate) range
    V_ent_max  = -0.22

    # --- Physical parameters (실험 조건 확인) ---
    f_drive  = 1.3e8             # pump drive frequency (Hz) — MUX: 130 MHz
    amp_gain = 1e-9              # Ithaco preamp gain (A/V) — 다이얼과 일치

    # ===== [드물게 수정] AL 알고리즘 파라미터 ===============================

    # --- Grid resolution ---
    n_exit = 200          # V_exit grid points (columns)
    n_ent  = 200          # V_ent grid points (rows)

    # --- AL parameters (Schoinas 2024 defaults) ---
    N_coarse     = 400    # Initial coarse scan points
    N_meas       = 60     # Measurement points per AL batch
    N_candidates = 100    # Candidate pool size per AL round
    N_AL         = 20     # Max AL iterations
    k_neighbors  = 4      # k for k-NN (paper: k=4)

    # --- eta extrapolation thresholds ---
    eta_noise = -3.5      # noise floor: eta < this is noise-dominated
    eta_max   = -0.6      # plateau boundary: eta > this is outside plateau

    # ===== [고정] 물리 상수 / 처리 파라미터 =================================

    e_charge = 1.602176634e-19   # elementary charge (C)
    I_sat_nA = 5.0               # ML soft saturation threshold (skill §5 Path 2)


## Instrument Controller (Skill §2, §3, §4, §7, §8)

Channel assignment (**CRITICAL — skill §2.1**):
- GPIB 8 = **V_exit** (exit gate)
- GPIB 1 = **V_ent** (entrance gate)
- GPIB 19 = **DMM** (Keithley 2000)

In [ ]:
class InstrumentController:
    """Encapsulates all hardware state (skill §7).

    Channel wiring (skill §2.1 — DO NOT CHANGE):
        GPIB43::8  = V_exit (exit gate)
        GPIB43::1  = V_ent  (entrance gate)
        GPIB43::19 = DMM    (Keithley 2000)
    """

    MAX_STEP_V   = 0.004   # 4 mV max voltage step (skill §3)
    RAMP_SETTLE  = 0.01    # 10 ms between ramp steps
    MEAS_SETTLE  = 0.1     # 100 ms before DMM read (Ithaco 1 nA/V, BW~30 Hz, ~22τ settling)
    DMM_RETRIES  = 3       # retry attempts (skill §8)
    DMM_TIMEOUT  = 10000   # 10 s timeout (skill §8)

    def __init__(self, config):
        self.config = config
        self.sim_mode = config.sim_mode
        self.gain = config.amp_gain           # A/V (skill §6)
        self.current_V_ent  = 0.0
        self.current_V_exit = 0.0
        self._total_meas = 0
        self._log_fh   = None   # scan history log file handle (Rule No.1 cross-check)
        self._log_path = None   # scan history log file path
        self.rm = None
        self.yoko_exit = None   # GPIB 8
        self.yoko_ent  = None   # GPIB 1
        self.dmm       = None   # GPIB 19

    # ── connect / disconnect ──────────────────────────────

    def connect(self):
        # -- open scan history log (survives abort via immediate flush) --
        _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        self._log_path = f"log_scanhistory_{_ts}.txt"
        self._log_fh = open(self._log_path, 'w', encoding='utf-8')
        self._log_fh.write("# ALSM voltage scan history — skill Rule No.1 (4 mV) cross-check\n")
        self._log_fh.write(f"# Created  : {datetime.now().isoformat()}\n")
        self._log_fh.write(f"# sim_mode : {self.sim_mode}\n")
        self._log_fh.write(f"# MAX_STEP : {self.MAX_STEP_V*1000:.1f} mV\n")
        self._log_fh.write(
            "# Columns  : timestamp  step_i/n  V_ent[V]  V_exit[V]"
            "  dV_ent[mV]  dV_exit[mV]  event\n")
        self._log_fh.flush()

        if self.sim_mode:
            print("[SIM] Simulation mode — no hardware connected")
            self._log_write("1/1", 0.0, 0.0, 0.0, 0.0, "SIM_CONNECT")
            return
        if not PYVISA_AVAILABLE:
            raise RuntimeError("pyvisa not installed — cannot connect")

        self.rm = pyvisa.ResourceManager()
        print(self.rm.list_resources())

        # V_exit = GPIB 8  (skill §2.1)
        self.yoko_exit = self.rm.open_resource(
            "GPIB43::8::INSTR",
            write_termination='\n', read_termination='\n')
        # V_ent = GPIB 1   (skill §2.1)
        self.yoko_ent = self.rm.open_resource(
            "GPIB43::1::INSTR",
            write_termination='\n', read_termination='\n')
        # DMM = GPIB 19
        self.dmm = self.rm.open_resource(
            "GPIB43::19::INSTR",
            write_termination='\n', read_termination='\n')
        self.dmm.timeout = self.DMM_TIMEOUT

        # Initialize Yokogawa outputs
        self.yoko_exit.write("F1R5O1E")
        self.yoko_ent.write("F1R5O1E")

        # Set DMM NPLC — try multiple SCPI variants (skill §8)
        for cmd in [":SENS:VOLT:NPLC 1.0", "NPLC 1.0", ":NPLC 1.0"]:
            try:
                self.dmm.write(cmd)
                break
            except Exception:
                pass

        # -- 안전 초기화: OD 쿼리로 실제 기기 전압을 읽어 추적 변수 동기화
        # (skill §3 Rule No.1 준수: S0.000000E를 직접 쓰면 이전 실행이
        #  비정상 종료된 경우 단일 명령으로 4 mV 초과 점프가 발생함)
        time.sleep(0.3)   # F1R5O1E 안정화
        try:
            V_ent_actual  = self._query_yoko_voltage(self.yoko_ent)
            V_exit_actual = self._query_yoko_voltage(self.yoko_exit)
        except Exception as e:
            print(f"[HW] FATAL: cannot query Yokogawa voltages: {e}")
            print("[HW] -> manually set both Yokogawas to 0 V, then reconnect.")
            if self._log_fh is not None:
                self._log_fh.write(f"# OD query failed: {e}\n")
                self._log_fh.close()
                self._log_fh = None
            raise RuntimeError("Yokogawa OD query failed — cannot determine safe start state")

        self.current_V_ent  = V_ent_actual
        self.current_V_exit = V_exit_actual
        print(f"[HW] Queried voltages: V_ent={V_ent_actual:+.4f} V, "
              f"V_exit={V_exit_actual:+.4f} V")
        self._log_write("0/0", V_ent_actual, V_exit_actual,
                        0.0, 0.0, "HW_CONNECT_QUERIED")

        # -- 현재 전압 != 0 이면 4 mV 스텝 램프로 안전하게 0 V 로 이동 --
        if abs(V_ent_actual) > 1e-6 or abs(V_exit_actual) > 1e-6:
            max_d_mV = max(abs(V_ent_actual), abs(V_exit_actual)) * 1000
            n_ramp = int(np.ceil(max_d_mV / (self.MAX_STEP_V * 1000)))
            print(f"[HW] Ramping to 0 V in {n_ramp} steps (<=4 mV each) ...")
            self.set_voltages(0.0, 0.0)   # 4 mV 규약 엄수 경로
            # 검증 쿼리
            V_chk_ent  = self._query_yoko_voltage(self.yoko_ent)
            V_chk_exit = self._query_yoko_voltage(self.yoko_exit)
            if abs(V_chk_ent) > 0.001 or abs(V_chk_exit) > 0.001:
                print(f"[HW] WARNING: ramp incomplete — "
                      f"V_ent={V_chk_ent:+.4f}, V_exit={V_chk_exit:+.4f}")

        print("[HW] Connected: V_exit=GPIB8, V_ent=GPIB1, DMM=GPIB19 — ready at 0 V")
        print(f"[LOG] Scan history: {self._log_path}")

    def ramp_to_safe(self):
        """Ramp all channels to 0 V (skill §4)."""
        self._log_write("0/0", self.current_V_ent, self.current_V_exit,
                         0.0, 0.0, "SAFE_RAMP_START")
        self.set_voltages(0.0, 0.0)
        print("[SAFE] All channels ramped to 0 V")

    def close(self):
        """Safe shutdown: ramp to 0 V then disconnect (skill §4).

        Robust against partial-connect failure: if GPIB resources are
        open, re-queries actual voltage via OD before ramping so that
        stale tracking variables (e.g. still at __init__ default 0.0
        after connect() raised partway) do not cause unsafe assumptions.
        """
        # Refresh tracking from actual instrument state (in case of
        # partial-connect failure where tracking is stale)
        if (not self.sim_mode and
                self.yoko_ent is not None and self.yoko_exit is not None):
            try:
                q_ent  = self._query_yoko_voltage(self.yoko_ent)
                q_exit = self._query_yoko_voltage(self.yoko_exit)
                if (abs(q_ent  - self.current_V_ent)  > 0.001 or
                    abs(q_exit - self.current_V_exit) > 0.001):
                    print(f"[CLOSE] Tracking refreshed from OD: "
                          f"V_ent {self.current_V_ent:+.4f}->{q_ent:+.4f}, "
                          f"V_exit {self.current_V_exit:+.4f}->{q_exit:+.4f}")
                self.current_V_ent  = q_ent
                self.current_V_exit = q_exit
            except Exception as e:
                print(f"[CLOSE] OD query failed ({e}); "
                      f"using last-known tracking values")
        self.ramp_to_safe()
        if self._log_fh is not None:
            self._log_fh.write(f"# Closed: {datetime.now().isoformat()}\n")
            self._log_fh.close()
            self._log_fh = None
            print(f"[LOG] Scan history saved: {self._log_path}")
        if not self.sim_mode:
            for inst in [self.yoko_ent, self.yoko_exit, self.dmm]:
                if inst is not None:
                    try:
                        inst.close()
                    except Exception:
                        pass
            if self.rm is not None:
                self.rm.close()
        print("[CLOSED] All instruments disconnected")

    # ── voltage control ───────────────────────────────────

    def set_voltages(self, V_ent, V_exit):
        """Ramp both channels simultaneously in <= 4 mV steps (skill §3).

        Both channels are interpolated in the same number of steps
        so they move together at each sub-step.
        """
        target = np.array([V_ent, V_exit])
        start  = np.array([self.current_V_ent, self.current_V_exit])
        max_delta = np.max(np.abs(target - start))
        n_steps = max(1, int(np.ceil(max_delta / self.MAX_STEP_V)))

        prev_vk = start.copy()
        for k in range(1, n_steps + 1):
            alpha = k / n_steps
            vk = start + alpha * (target - start)
            if not self.sim_mode:
                # Write both channels at each step (skill §3)
                self.yoko_ent.write(f'S{vk[0]:.6f}E')
                self.yoko_exit.write(f'S{vk[1]:.6f}E')
                time.sleep(self.RAMP_SETTLE)
            self._log_write(f"{k}/{n_steps}",
                            vk[0], vk[1],
                            (vk[0]-prev_vk[0])*1000,
                            (vk[1]-prev_vk[1])*1000,
                            "ramp")
            prev_vk = vk.copy()

        self.current_V_ent  = V_ent
        self.current_V_exit = V_exit

    # ── measurement ───────────────────────────────────────

    def measure(self, V_ent, V_exit):
        """Set voltages and read DMM.

        Returns RAW DMM voltage (skill §5 Path 1).
        Physical current: I[A] = raw_V * self.gain
        """
        self.set_voltages(V_ent, V_exit)
        self._total_meas += 1

        if self.sim_mode:
            return self.config.sim_func(V_ent, V_exit)

        time.sleep(self.MEAS_SETTLE)
        for attempt in range(self.DMM_RETRIES):
            try:
                return float(self.dmm.query("fetch?"))
            except Exception as e:
                if attempt < self.DMM_RETRIES - 1:
                    time.sleep(0.5)
                else:
                    print(f"[DMM] Failed after {self.DMM_RETRIES} attempts: {e}")
                    return np.nan

    # ── gain management (skill §6) ───────────────────────

    # ── Yokogawa query helper ─────────────────────────────

    def _query_yoko_voltage(self, yoko):
        """Query current output voltage of a Yokogawa 7651 via OD command.

        OD is a read-only query — does NOT change instrument state.
        Typical response: 'NDCV+0.5000E+00'. Regex also accepts
        leading-zero-less '+.5000E+00' and integer forms for firmware
        variants. Returns voltage in volts.
        """
        resp = yoko.query("OD").strip()
        m = re.search(r'([+-]?(?:\d+\.?\d*|\.\d+)(?:[eE][+-]?\d+)?)', resp)
        if m is None:
            raise RuntimeError(f"Cannot parse Yokogawa OD response: {resp!r}")
        return float(m.group(1))

    # ── scan history logging ─────────────────────────────

    def _log_write(self, step_label, V_ent, V_exit, dV_ent_mV, dV_exit_mV, event):
        """Write one line to scan history log and flush immediately.

        Flushing after every write ensures the log is preserved even
        if the kernel is killed mid-measurement (abort safety).
        """
        if self._log_fh is None:
            return
        ts = datetime.now().isoformat(timespec='milliseconds')
        line = (f"{ts}\t{step_label}\t"
                f"{V_ent:+.6f}\t{V_exit:+.6f}\t"
                f"{dV_ent_mV:+.3f}\t{dV_exit_mV:+.3f}\t{event}\n")
        self._log_fh.write(line)
        self._log_fh.flush()   # immediate flush — abort-safe

    def set_gain(self, new_gain):
        """Set Ithaco gain with human confirmation (skill §6.3).
        Skips prompt in simulation mode."""
        if not self.sim_mode:
            print(f"\n*** MANUAL GAIN CHANGE REQUIRED ***")
            print(f"Set Ithaco dial to: {new_gain:.0e} A/V")
            entered = float(input("Confirm gain value: "))
            if abs(entered - new_gain) / new_gain > 0.01:
                raise RuntimeError("Gain mismatch — aborting")
            time.sleep(2.0)   # preamp settling
        self.gain = float(new_gain)

    # ── unit conversion helpers ───────────────────────────

    def raw_to_current_A(self, raw_V):
        """Convert raw DMM voltage to current in Amperes."""
        return raw_V * self.gain

    def raw_to_n(self, raw_V):
        """Convert raw DMM voltage to <n> = I / (e*f)."""
        I_A = self.raw_to_current_A(raw_V)
        ef = self.config.e_charge * self.config.f_drive
        return I_A / ef

    @property
    def total_measurements(self):
        return self._total_meas

## k-NN Active Learner (Schoinas 2024)

Single k-NN (k=4) with **neighbor variance** as uncertainty proxy.

- Prediction: inverse-distance weighted mean of k nearest neighbors
- Uncertainty: variance of k neighbors' measured values
- High variance = neighbors disagree = likely near plateau boundary = measure here

In [ ]:
class KNN_ActiveLearner:
    """Single k-NN regressor with neighbor variance for AL.

    Following Schoinas 2024 (APL 125, 124001):
    - k=4 nearest neighbors, inverse-distance weighted prediction
    - Neighbor variance as AL acquisition function (uncertainty proxy)
    """

    def __init__(self, k=4):
        self.k = k
        self.X_train = None
        self.y_train = None
        self._nn = None

    def fit(self, X, y):
        """Fit model with training data."""
        self.X_train = np.copy(X)
        self.y_train = np.copy(y)
        k_use = min(self.k, len(X))
        self._nn = NearestNeighbors(n_neighbors=k_use, algorithm='auto')
        self._nn.fit(X)

    def predict(self, X_test):
        """Predict at test points.

        Returns
        -------
        y_pred : (N,) inverse-distance weighted mean
        y_var  : (N,) variance of k neighbors' y-values (AL uncertainty)
        """
        k_use = min(self.k, len(self.X_train))
        distances, indices = self._nn.kneighbors(X_test, n_neighbors=k_use)
        y_neighbors = self.y_train[indices]       # (N_test, k)

        # Inverse-distance weighted prediction
        weights = 1.0 / (distances + 1e-10)
        weights = weights / weights.sum(axis=1, keepdims=True)
        y_pred = np.sum(weights * y_neighbors, axis=1)

        # Neighbor variance — AL uncertainty proxy
        y_var = np.var(y_neighbors, axis=1)

        return y_pred, y_var

    @property
    def n_train(self):
        return 0 if self.X_train is None else len(self.X_train)

## Simulation & Replay Functions

Two simulator modes are available as drop-in replacements for hardware:

| Function | Purpose |
|----------|---------|
| `sim_pump_plateau(V_ent, V_exit)` | Analytic sigmoid plateau — no data needed, fast |
| `sim_generic(V_ent, V_exit)` | Generic math test functions for AL algorithm testing |
| `make_replay_sim(filepath)` | **Real-data replay** — loads measured (V_ent, V_exit, I[nA]) file and bilinearly interpolates |

Both return a raw DMM voltage compatible with `InstrumentController.measure()`.

In [ ]:
def sim_pump_plateau(V_ent, V_exit):
    """Simulated pump n=1 plateau for eta_E testing.

    Returns <n> = I/(ef).  The plateau shape is a product of two
    sigmoids (loading × emission) whose position shifts with V_ent.

    V_exit range: ~ [-0.5, -0.22]
    V_ent  range: ~ [-0.5, -0.22]
    """
    V_exit_center = -0.36
    V_ent_center  = -0.36
    width_exit = 0.015
    shift = (V_ent - V_ent_center) * 0.4

    loading  = 1.0 / (1.0 + np.exp(-(V_exit - (V_exit_center - 0.06 + shift)) / width_exit))
    emission = 1.0 / (1.0 + np.exp( (V_exit - (V_exit_center + 0.06 + shift)) / width_exit))
    return loading * emission + np.random.normal(0, 1e-6)


def sim_generic(V_ent, V_exit, icase=6):
    """Generic test functions for AL algorithm testing (from QMT notebook)."""
    x, y = V_exit, V_ent
    if icase == 6:  return np.cos(0.1 * x) * np.cos(0.2 * y)
    if icase == 7:  return np.sin(0.1 * x) ** 2 + np.cos(0.2 * y) ** 2
    if icase == 8:  return np.cos(0.1 * x + np.sin(0.2 * y))
    if icase == 9:  return np.cos(0.1 * x) + np.sin(0.2 * y)
    return 0.0


def make_replay_sim(filepath, amp_gain=1e-9):
    """Create a real-data replay simulator.

    Loads a (V_ent, V_exit, I[nA]) measurement file and returns a
    callable that bilinearly interpolates the data on demand.

    The returned function outputs raw DMM voltage [V] = I[A] / amp_gain,
    which is exactly what InstrumentController.measure() returns in
    hardware mode. Set cfg.amp_gain to the same value.

    Parameters
    ----------
    filepath : str, path to data file (columns: V_ent  V_exit  I[nA])
    amp_gain : float, Ithaco gain (A/V) — MUST match cfg.amp_gain
               With amp_gain=1e-9 A/V: raw output = I[nA] numerically.

    Returns
    -------
    replay_sim : callable(V_ent, V_exit) → float (raw DMM voltage, V)
    info       : dict with data range and grid statistics

    Notes
    -----
    - RegularGridInterpolator requires a regular (V_ent × V_exit) grid.
      Missing cells are filled by nearest-neighbor before interpolation.
    - bounds_error=False, fill_value=None: extrapolates at edges via
      nearest boundary value (safe for small out-of-range queries).
    """
    from scipy.interpolate import RegularGridInterpolator, NearestNDInterpolator

    # Load raw data (col0=V_ent, col1=V_exit, col2=I[nA])
    data        = np.loadtxt(filepath)
    V_ent_d     = data[:, 0]
    V_exit_d    = data[:, 1]
    I_nA_d      = data[:, 2]

    # Build sorted unique axes
    v_ent_u  = np.sort(np.unique(np.round(V_ent_d,  8)))
    v_exit_u = np.sort(np.unique(np.round(V_exit_d, 8)))
    n_ent_d  = len(v_ent_u)
    n_exit_d = len(v_exit_u)

    # Index maps (round to avoid float precision issues)
    ent_idx  = {round(v, 8): i for i, v in enumerate(v_ent_u)}
    exit_idx = {round(v, 8): i for i, v in enumerate(v_exit_u)}

    # Fill I grid (n_ent × n_exit)
    I_grid = np.full((n_ent_d, n_exit_d), np.nan)
    for ve, vx, I in zip(V_ent_d, V_exit_d, I_nA_d):
        ii = ent_idx.get(round(ve, 8))
        jj = exit_idx.get(round(vx, 8))
        if ii is not None and jj is not None:
            I_grid[ii, jj] = I

    # Fill any NaN cells by nearest-neighbor interpolation
    nan_mask = np.isnan(I_grid)
    if nan_mask.any():
        known_pts  = [(v_ent_u[i], v_exit_u[j])
                      for i in range(n_ent_d) for j in range(n_exit_d)
                      if not nan_mask[i, j]]
        known_vals = [I_grid[i, j]
                      for i in range(n_ent_d) for j in range(n_exit_d)
                      if not nan_mask[i, j]]
        nn_fill = NearestNDInterpolator(known_pts, known_vals)
        for i in range(n_ent_d):
            for j in range(n_exit_d):
                if nan_mask[i, j]:
                    I_grid[i, j] = float(nn_fill([[v_ent_u[i], v_exit_u[j]]])[0])

    # Build bilinear RegularGridInterpolator
    # axes: (V_ent, V_exit),  values: I[nA]
    interp = RegularGridInterpolator(
        (v_ent_u, v_exit_u), I_grid,
        method='linear', bounds_error=False, fill_value=None)

    info = {
        'filepath':     os.path.basename(filepath),
        'n_ent':        n_ent_d,
        'n_exit':       n_exit_d,
        'n_pts':        len(data),
        'v_ent_range':  [float(v_ent_u.min()),  float(v_ent_u.max())],
        'v_exit_range': [float(v_exit_u.min()), float(v_exit_u.max())],
        'I_nA_range':   [float(I_nA_d.min()),   float(I_nA_d.max())],
        'amp_gain':     amp_gain,
    }
    print(f"[REPLAY] {info['filepath']}")
    print(f"  V_ent  : {info['v_ent_range']} V  ({n_ent_d} lines)")
    print(f"  V_exit : {info['v_exit_range']} V  ({n_exit_d} pts)")
    print(f"  I      : [{I_nA_d.min():.4f}, {I_nA_d.max():.4f}] nA")
    print(f"  Total  : {len(data)} points")

    def replay_sim(V_ent_q, V_exit_q):
        """Interpolate real data → raw DMM voltage (V) = I[A] / amp_gain."""
        I_val = float(interp([[V_ent_q, V_exit_q]])[0])
        return I_val * 1e-9 / amp_gain   # I[nA]×1e-9 / (A/V) = V

    return replay_sim, info

## Path Optimization & File I/O

In [ ]:
def greedy_nearest_neighbor_order(points, start_point=None):
    """Order points by greedy nearest-neighbor to minimize total voltage travel.

    This replaces the paper's random-jump measurement order with a
    physically safe traversal that keeps voltage steps small.
    """
    n = len(points)
    if n <= 1:
        return np.arange(n)

    visited = np.zeros(n, dtype=bool)
    order   = np.zeros(n, dtype=int)

    if start_point is not None:
        dists = np.linalg.norm(points - start_point, axis=1)
        order[0] = np.argmin(dists)
    else:
        order[0] = 0
    visited[order[0]] = True

    for i in range(1, n):
        current = points[order[i - 1]]
        dists = np.linalg.norm(points - current, axis=1)
        dists[visited] = np.inf
        order[i] = np.argmin(dists)
        visited[order[i]] = True

    return order


def write_accum_header(filepath, config, controller):
    """Write metadata header to accumulation file (skill §6.4, §10)."""
    with open(filepath, 'w') as f:
        f.write(f"# ALSM measurement data\n")
        f.write(f"# date={datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"# gain={controller.gain:.0e} A/V\n")
        f.write(f"# ef={config.e_charge * config.f_drive:.6e} A\n")
        f.write(f"# V_exit_range=[{config.V_exit_min}, {config.V_exit_max}]\n")
        f.write(f"# V_ent_range=[{config.V_ent_min}, {config.V_ent_max}]\n")
        f.write(f"# columns: V_ent  V_exit  y_raw\n")


def append_to_accum(filepath, V_ent, V_exit, y_raw):
    """Append one measurement to accumulation file (raw value, skill §5 Path 1)."""
    with open(filepath, 'a') as f:
        f.write(f"{V_ent}  {V_exit}  {y_raw}\n")

## $\eta$ and $\eta_E$ Computation

Convention: $\eta = \log_{10}(|\langle n \rangle - 1|)$ — more negative = better quantization.

$\eta_E$ extrapolation uses the **nonlinear Schoinas model** (skill §2, §3 — linear fitting is prohibited):
$$\eta(V) = \log_{10}\!\left(10^{a_1 V + b_1} + 10^{a_2 V + b_2}\right)$$
- Left asymptote $a_1 < 0$ (loading side), right asymptote $a_2 > 0$ (emission side)
- $\eta_E^{\min} = a_1 V_\mathrm{opt} + b_1$ where $V_\mathrm{opt} = (b_2 - b_1)/(a_1 - a_2)$
- **Mathematically guaranteed**: $\eta_E^{\min} <$ any measured $\eta$ value

$\eta_\text{noise}$ is estimated as the **10th percentile of actual measured data** (skill §4).  
Must NOT use interpolated/GPR-smoothed data — smoothing attenuates noise and gives a falsely shallow floor.

In [ ]:
def find_eta_noise(y_train, gain, ef, eta_max=-0.6):
    """Estimate η_noise as 10th percentile of measured η (skill §4).

    MUST use actual measured data — NOT interpolated or GPR-smoothed data.
    Smoothing attenuates noise, giving a falsely shallow η_noise.

    Parameters
    ----------
    y_train : 1D array, raw DMM voltage measurements
    gain    : float, Ithaco preamp gain (A/V)
    ef      : float, e * f (A)
    eta_max : upper cutoff to exclude transition region (default -0.6)

    Returns
    -------
    eta_noise : float, 10th-percentile η from actual measurements
    """
    n_vals   = y_train * gain / ef
    eta_vals = np.log10(np.clip(np.abs(n_vals - 1.0), 1e-15, None))
    valid    = eta_vals[np.isfinite(eta_vals) & (eta_vals < eta_max)]
    if len(valid) < 3:
        return eta_max
    return float(np.percentile(valid, 10))


def compute_eta(n_map):
    """Compute η = log₁₀(|⟨n⟩ − 1|) from ⟨n⟩ map.

    Parameters
    ----------
    n_map : 2D array of ⟨n⟩ = I/(ef) values.

    Returns
    -------
    eta_map : 2D array, same shape. More negative = better quantization.
    """
    deviation = np.abs(n_map - 1.0)
    deviation = np.clip(deviation, 1e-15, None)    # avoid log(0)
    return np.log10(deviation)


def _schoinas_model(V, a1, b1, a2, b2):
    """Nonlinear Schoinas η model (skill §2).

    η(V) = log₁₀(10^(a₁V+b₁) + 10^(a₂V+b₂))

    Numerically stable log-sum-exp formulation:
      η = max(L, R) + log₁₀(10^(L−max) + 10^(R−max))

    Left asymptote:  η → a₁V + b₁  (a₁ < 0, loading side)
    Right asymptote: η → a₂V + b₂  (a₂ > 0, emission side)
    """
    L  = a1 * V + b1
    R  = a2 * V + b2
    mx = np.maximum(L, R)
    return mx + np.log10(10.0 ** (L - mx) + 10.0 ** (R - mx))


def extrapolate_eta_line(v_exit, eta_values, eta_noise=-3.5, eta_max=-0.6):
    """Extrapolate η_E using the nonlinear Schoinas model (skill §2, §3).

    The former double-linear-fit approach is PROHIBITED (skill §3):
    linear fits attenuate asymptote slopes and produce η_E < η_min in
    only 9% of cases (Monte Carlo over sparse sampling, 100 trials).

    Model:  η(V) = log₁₀(10^(a₁V+b₁) + 10^(a₂V+b₂))
    η_E^min = a₁·V_opt + b₁   where  V_opt = (b₂−b₁)/(a₁−a₂)
    Mathematically guaranteed: η_E^min < min(η_data)

    Parameters
    ----------
    v_exit     : 1D array of V_exit values
    eta_values : 1D array of η at each V_exit
    eta_noise  : kept for API compatibility; not used in fit selection
    eta_max    : upper cutoff — exclude points outside plateau (η > eta_max)

    Returns
    -------
    eta_E    : η_E^min (scalar or np.nan on failure)
    fit_info : dict with model parameters, curve, residual, asymptotes
    """
    # Fit region: below plateau boundary (skill §2 — no lower bound needed)
    fit_mask = np.isfinite(eta_values) & (eta_values < eta_max)
    if fit_mask.sum() < 4:
        return np.nan, {}

    V_fd   = v_exit[fit_mask]
    eta_fd = eta_values[fit_mask]

    # Initial estimates via polyfit on each side of the minimum
    i_cen  = np.argmin(eta_fd)
    V_cen  = V_fd[i_cen]
    left   = V_fd <= V_cen
    right  = V_fd >= V_cen

    sL_i, bL_i = (np.polyfit(V_fd[left],  eta_fd[left],  1)
                  if left.sum()  >= 2 else (-1.0, eta_fd.mean()))
    sR_i, bR_i = (np.polyfit(V_fd[right], eta_fd[right], 1)
                  if right.sum() >= 2 else ( 1.0, eta_fd.mean()))

    try:
        popt, _ = curve_fit(_schoinas_model, V_fd, eta_fd,
                            p0=[sL_i, bL_i, sR_i, bR_i], maxfev=10000)
    except RuntimeError:
        return np.nan, {}

    a1, b1, a2, b2 = popt

    # Validate: asymptote slopes must be physically correct
    if a1 >= 0 or a2 <= 0:
        return np.nan, {}

    # η_E^min at asymptote intersection (skill §2)
    V_opt  = (b2 - b1) / (a1 - a2)
    eta_E  = a1 * V_opt + b1

    # Model curve and RMS residual
    V_model   = np.linspace(v_exit.min(), v_exit.max(), 300)
    eta_model = _schoinas_model(V_model, *popt)
    rms       = float(np.sqrt(np.mean(
        (_schoinas_model(V_fd, *popt) - eta_fd) ** 2)))

    fit_info = {
        'v_opt': V_opt,   'eta_E': eta_E,
        'a1': a1, 'b1': b1, 'a2': a2, 'b2': b2,
        'V_model': V_model, 'eta_model': eta_model,
        'fit_mask': fit_mask, 'rms': rms,
        # Backward-compat aliases (used by extrapolate_eta_map, plot_fig1f)
        'v_cross': V_opt, 'slope_L': a1, 'intercept_L': b1,
        'slope_R': a2,    'intercept_R': b2,
    }
    return eta_E, fit_info


def extrapolate_eta_map(eta_map, v_exit_grid, v_ent_grid,
                        eta_noise=-3.5, eta_max=-0.6):
    """Compute 2D η_E(V_exit, V_ent) map via Schoinas extrapolation along V_exit
    for each V_ent value.

    Parameters
    ----------
    eta_map    : (n_ent, n_exit) array of η values
    v_exit_grid: (n_exit,) V_exit values
    v_ent_grid : (n_ent,) V_ent values

    Returns
    -------
    eta_E_map  : (n_ent, n_exit) asymptotic-envelope map
                 (each row = max(a₁V+b₁, a₂V+b₂) for that V_ent)
    eta_E_1d   : (n_ent,) η_E^min as function of V_ent
    fit_infos  : list of fit_info dicts, one per V_ent row
    """
    n_ent  = len(v_ent_grid)
    n_exit = len(v_exit_grid)
    eta_E_1d  = np.full(n_ent, np.nan)
    fit_infos = []

    for i in range(n_ent):
        # eta_map[i, :] = η along V_exit at fixed V_ent = v_ent_grid[i]
        eta_line = eta_map[i, :]
        eta_E, info = extrapolate_eta_line(
            v_exit_grid, eta_line, eta_noise=eta_noise, eta_max=eta_max)
        eta_E_1d[i] = eta_E
        fit_infos.append(info)

    # 2D map: asymptotic envelope max(a₁V+b₁, a₂V+b₂) per row
    eta_E_map = np.full((n_ent, n_exit), np.nan)
    for i in range(n_ent):
        info = fit_infos[i]
        if not info:
            continue
        line_L = info['a1'] * v_exit_grid + info['b1']
        line_R = info['a2'] * v_exit_grid + info['b2']
        eta_E_map[i, :] = np.maximum(line_L, line_R)

    return eta_E_map, eta_E_1d, fit_infos

## ALSM Driver

Combined algorithm:
1. **Coarse scan**: $N_\text{coarse}$ points on uniform sub-grid, greedy nearest-neighbor path
2. **AL loop**: k-NN variance → select top $N_\text{candidates}$ → sample $N_\text{meas}$ → measure → retrain
3. **Interpolation**: 2D piecewise linear (scipy `griddata`) on fine grid
4. **Extrapolation**: $\eta_E$ via nonlinear Schoinas model (skill §2 — double-linear fit is prohibited)

Meshgrid convention (skill §2.3):
```
rows  i  → V_ent   (d)
cols  j  → V_exit  (b)
```

In [ ]:
def driver_alsm(config, controller):
    """Main ALSM driver.

    Parameters
    ----------
    config     : ALSMConfig instance
    controller : InstrumentController instance (already connected)

    Returns
    -------
    results : dict with all outputs for visualization
    """

    # ── Setup ─────────────────────────────────────────────
    timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = f'output_ALSM_{timestamp}'
    os.makedirs(output_dir, exist_ok=True)
    accum_path = os.path.join(output_dir, 'accum.txt')
    write_accum_header(accum_path, config, controller)

    print(f"Output : {output_dir}")
    print(f"V_exit : [{config.V_exit_min}, {config.V_exit_max}]")
    print(f"V_ent  : [{config.V_ent_min}, {config.V_ent_max}]")
    print(f"Grid   : {config.n_ent} x {config.n_exit}")
    print(f"AL     : N_coarse={config.N_coarse}, N_meas={config.N_meas}, "
          f"N_AL={config.N_AL}, k={config.k_neighbors}")

    start_time = time.time()

    # ── Build fine grid (skill §2.3 meshgrid convention) ──
    # b = V_exit (columns), d = V_ent (rows)
    b = np.linspace(config.V_exit_min, config.V_exit_max, config.n_exit)
    d = np.linspace(config.V_ent_min,  config.V_ent_max,  config.n_ent)
    B, D = np.meshgrid(b, d)
    # grid_points[i] = (V_ent, V_exit) — row-major order
    # B[i,j] = b[j] = V_exit,  D[i,j] = d[i] = V_ent
    grid_points = np.column_stack([D.ravel(), B.ravel()])   # (n_total, 2)
    n_total = len(grid_points)

    measured_mask   = np.zeros(n_total, dtype=bool)
    measured_values = np.full(n_total, np.nan)

    X_list = []   # (V_ent, V_exit) pairs
    y_list = []   # raw measured values

    # ── Soft saturation for ML training (skill §5 Path 2) ─
    I_sat_raw = config.I_sat_nA * 1e-9 / controller.gain   # convert to raw DMM volts
    def soft_sat(y_raw):
        return I_sat_raw * np.tanh(y_raw / I_sat_raw)

    # ── Step 1: Coarse scan ───────────────────────────────
    n_side_ent  = int(np.ceil(np.sqrt(config.N_coarse * config.n_ent / config.n_exit)))
    n_side_exit = int(np.ceil(config.N_coarse / n_side_ent))
    idx_ent  = np.linspace(0, config.n_ent  - 1, n_side_ent,  dtype=int)
    idx_exit = np.linspace(0, config.n_exit - 1, n_side_exit, dtype=int)

    coarse_flat = []
    for i_ent in idx_ent:
        for j_exit in idx_exit:
            coarse_flat.append(i_ent * config.n_exit + j_exit)
    coarse_flat = np.array(coarse_flat)
    coarse_points = grid_points[coarse_flat]

    # Order by greedy nearest-neighbor from current position
    start_pos = np.array([controller.current_V_ent, controller.current_V_exit])
    order = greedy_nearest_neighbor_order(coarse_points, start_point=start_pos)
    coarse_flat   = coarse_flat[order]
    coarse_points = coarse_points[order]

    print(f"\n--- Coarse scan: {len(coarse_points)} points ---")
    for pt, idx in zip(coarse_points, coarse_flat):
        val = controller.measure(pt[0], pt[1])   # pt = (V_ent, V_exit)
        measured_mask[idx]   = True
        measured_values[idx] = val
        X_list.append(pt)
        y_list.append(val)
        append_to_accum(accum_path, pt[0], pt[1], val)

    X_train = np.array(X_list)
    y_train = np.array(y_list)
    print(f"Coarse done: {len(y_train)} points, "
          f"{time.time() - start_time:.1f}s")

    # ── Step 2: Initialize k-NN model ─────────────────────
    model = KNN_ActiveLearner(k=config.k_neighbors)
    # ML training uses soft-saturated values (skill §5 Path 2)
    model.fit(X_train, soft_sat(y_train))
    print(f"Model: k-NN (k={config.k_neighbors}), neighbor variance")

    # ── Step 3: Active Learning loop ──────────────────────
    al_history = []
    snapshot_round = 3   # save variance map at this round for Fig.1(c)
    snapshot_data  = {}

    for al_round in range(config.N_AL):
        # Predict on all grid points (for variance map)
        y_pred_all, y_var_all = model.predict(grid_points)

        # Unmeasured points only
        unmeasured_idx = np.where(~measured_mask)[0]
        if len(unmeasured_idx) == 0:
            print("All grid points measured!")
            break

        y_var_unmeasured = y_var_all[unmeasured_idx]
        max_var = np.max(y_var_unmeasured)

        # Select top N_candidates by variance
        N_cand = min(config.N_candidates, len(unmeasured_idx))
        N_meas = min(config.N_meas, N_cand)
        top_cand_local = np.argsort(y_var_unmeasured)[-N_cand:]

        # Sample N_meas from candidates (uniform, per paper)
        selected_local    = np.random.choice(top_cand_local, size=N_meas, replace=False)
        selected_grid_idx = unmeasured_idx[selected_local]
        selected_points   = grid_points[selected_grid_idx]

        # Save snapshot at round 3 for Fig.1(c) visualization
        if al_round + 1 == snapshot_round:
            snapshot_data = {
                'variance_map': y_var_all.reshape(config.n_ent, config.n_exit),
                'selected_points': selected_points.copy(),
                'round': al_round + 1,
            }

        # Order by greedy nearest-neighbor
        current_pos = np.array([controller.current_V_ent, controller.current_V_exit])
        path_order = greedy_nearest_neighbor_order(selected_points, start_point=current_pos)
        selected_points   = selected_points[path_order]
        selected_grid_idx = selected_grid_idx[path_order]

        # Measure batch
        for pt, idx in zip(selected_points, selected_grid_idx):
            val = controller.measure(pt[0], pt[1])
            measured_mask[idx]   = True
            measured_values[idx] = val
            X_list.append(pt)
            y_list.append(val)
            append_to_accum(accum_path, pt[0], pt[1], val)

        # Retrain
        X_train = np.array(X_list)
        y_train = np.array(y_list)
        model.fit(X_train, soft_sat(y_train))

        total_meas = int(np.sum(measured_mask))
        elapsed = time.time() - start_time
        al_history.append((al_round + 1, total_meas, max_var, elapsed))

        print(f"AL {al_round+1:3d}/{config.N_AL}: "
              f"+{N_meas} pts, total={total_meas}, "
              f"max_var={max_var:.4e}, {elapsed:.1f}s")

    # ── Step 4: 2D interpolation (paper method) ───────────
    print(f"\n--- Post-processing ---")
    measured_idx = np.where(measured_mask)[0]

    # Piecewise linear interpolation (paper: scipy griddata)
    interp_values = griddata(
        grid_points[measured_idx], measured_values[measured_idx],
        grid_points, method='linear')

    # Fill NaN outside convex hull with k-NN predictions
    y_pred_final, _ = model.predict(grid_points)
    nan_mask = np.isnan(interp_values)      # outside convex hull → extrapolation region
    interp_values[nan_mask] = y_pred_final[nan_mask]

    # For measured points, always use actual values
    interp_values[measured_mask] = measured_values[measured_mask]

    # Reshape to 2D: (n_ent, n_exit) — rows=V_ent, cols=V_exit
    interp_map = interp_values.reshape(config.n_ent, config.n_exit)

    # ── Data-source counts (Task 1) ───────────────────────
    n_measured     = int(measured_mask.sum())
    n_extrapolated = int((nan_mask & ~measured_mask).sum())   # kNN fill (outside hull)
    n_interpolated = n_total - n_measured - n_extrapolated    # griddata linear (inside hull)
    assert n_measured + n_interpolated + n_extrapolated == n_total, \
        "grid count partition mismatch"

    total_time = time.time() - start_time
    print(f"\n--- Summary ---")
    print(f"Measured      : {n_measured:>6d} pts  ({100*n_measured/n_total:5.1f}%)")
    print(f"Interpolated  : {n_interpolated:>6d} pts  ({100*n_interpolated/n_total:5.1f}%)  [griddata linear, inside convex hull]")
    print(f"Extrapolated  : {n_extrapolated:>6d} pts  ({100*n_extrapolated/n_total:5.1f}%)  [k-NN fallback, outside convex hull]")
    print(f"Total (grid)  : {n_total:>6d} pts  ({config.n_ent}x{config.n_exit})")
    print(f"Reduction     : {n_total/max(n_measured,1):.1f}x fewer measurements")
    print(f"Elapsed time  : {total_time:.1f} s  ({total_time/60:.2f} min)")

    # ── Build config_parmeter dict (Task 2) ───────────────
    # hardware config + ALSM config 병합 — 재현용 메타데이터.
    # (user's spelling 'config_parmeter' preserved intentionally.)
    def _extract_public_attrs(obj):
        out = {}
        for k in dir(obj):
            if k.startswith('_'):
                continue
            try:
                v = getattr(obj, k)
            except Exception:
                continue
            if callable(v):
                continue
            # Only serialize plain scalars / strings / booleans / None
            if isinstance(v, (int, float, str, bool)) or v is None:
                out[k] = v
        return out

    config_parmeter = {
        'hardware': {
            'GPIB_V_exit'  : 'GPIB43::8::INSTR',
            'GPIB_V_ent'   : 'GPIB43::1::INSTR',
            'GPIB_DMM'     : 'GPIB43::19::INSTR',
            'amp_gain_A_per_V' : controller.gain,
            'MAX_STEP_V'   : controller.MAX_STEP_V,
            'RAMP_SETTLE_s': controller.RAMP_SETTLE,
            'MEAS_SETTLE_s': controller.MEAS_SETTLE,
            'DMM_RETRIES'  : controller.DMM_RETRIES,
            'DMM_TIMEOUT_ms': controller.DMM_TIMEOUT,
            'sim_mode'     : controller.sim_mode,
            'scan_history_log': getattr(controller, '_log_path', None),
        },
        'alsm': _extract_public_attrs(config),
        'run': {
            'timestamp'        : timestamp,
            'output_dir'       : output_dir,
            'n_measured'       : n_measured,
            'n_interpolated'   : n_interpolated,
            'n_extrapolated'   : n_extrapolated,
            'n_total_grid'     : n_total,
            'elapsed_seconds'  : total_time,
        },
    }

    # ── Save results ──────────────────────────────────────
    np.savez(os.path.join(output_dir, 'results.npz'),
             v_exit_grid=b, v_ent_grid=d,
             interp_map=interp_map,
             X_train=X_train, y_train=y_train,
             measured_mask=measured_mask.reshape(config.n_ent, config.n_exit),
             measured_values=measured_values.reshape(config.n_ent, config.n_exit),
             al_history=np.array(al_history) if al_history else np.array([]),
             gain=controller.gain,
             n_measured=n_measured,
             n_interpolated=n_interpolated,
             n_extrapolated=n_extrapolated,
             n_total_grid=n_total,
             elapsed_seconds=total_time,
             config_parmeter=np.array(config_parmeter, dtype=object))

    results = {
        'interp_map': interp_map,
        'X_train': X_train,
        'y_train': y_train,
        'v_exit_grid': b,
        'v_ent_grid': d,
        'measured_mask': measured_mask.reshape(config.n_ent, config.n_exit),
        'al_history': al_history,
        'snapshot_data': snapshot_data,
        'output_dir': output_dir,
        'total_time': total_time,
        'config': config,
        'gain': controller.gain,
        'n_measured'    : n_measured,
        'n_interpolated': n_interpolated,
        'n_extrapolated': n_extrapolated,
        'n_total_grid'  : n_total,
        'config_parmeter': config_parmeter,
    }
    return results

## Visualization — Paper Fig.1 Reproduction

| Function | Paper Figure | Content |
|----------|-------------|---------|
| `plot_fig1c` | Fig.1(c) | Variance heatmap at AL round 3 + selected points |
| `plot_fig1d` | Fig.1(d) | Sparse measured $\eta_M$ points after all AL iterations |
| `plot_fig1e` | Fig.1(e) | Interpolated $\eta_M$ map with $\eta_\text{noise}$ contour |
| `plot_fig1f` | Fig.1(f) | $\eta$ vs $V_\text{exit}$ line fit at fixed $V_\text{ent}$ |
| `plot_fig1g` | Fig.1(g) | 2D extrapolated $\eta_E(V_\text{exit}, V_\text{ent})$ map |

Axis labels: $V_\text{exit}$, $V_\text{ent}$ (skill §11.1 — never $V_1$, $V_2$)

In [ ]:
# -- Development plot style (_ST) -------------------------------------------
# lab-plot-style skill §15: development/operational notebook minimum standard.
# Modify values here; never hardcode in individual plot functions.
#
#   show_title = True   <- development mode (titles visible)
#   show_title = False  <- publication mode (match plot_output_ALSAM.ipynb)

_ST = dict(
    # Figure sizes (inches)
    figsize_sq   = (8, 7),    # single panel square-ish
    figsize_wide = (10, 6),   # single panel wide
    figsize_2col = (16, 6),   # 2-panel side-by-side
    figsize_hist = (12, 4),   # 2-panel histogram
    figsize_conv = (8, 5),    # convergence / diagnostic plot
    # Fonts
    fs_label  = 11,           # axis label pt
    fs_tick   = 9,            # tick label pt
    fs_legend = 9,            # legend pt
    fs_annot  = 10,           # annotation pt
    # Lines & markers
    lw        = 1.5,          # default linewidth
    lw_model  = 2.0,          # model curve
    lw_asym   = 1.2,          # asymptote / secondary lines
    ms_data   = 3,            # data point markersize
    ms_fit    = 5,            # fit point markersize
    ms_best   = 14,           # optimal point (star)
    # Colors
    c_data    = '#aaaaaa',    # raw data points (gray)
    c_fit     = 'red',        # fit / model elements
    c_best    = 'darkred',    # optimum marker
    c_noise   = 'gray',       # noise floor line
    c_conv    = 'steelblue',  # convergence / step plot lines
    c_step    = 'steelblue',  # histogram bars
    # Colormaps
    cmap_var  = 'hot_r',      # variance heatmap
    cmap_eta  = 'RdYlGn_r',   # eta heatmap
    cmap_n    = 'viridis',    # <n> heatmap
    # Control flags
    show_title  = True,       # True = dev mode; False = publication
    legend_loc  = 'upper right',
    legend_ncol = 2,
    annot_xy    = (0.55, 0.3), # axes-fraction xytext for annotate arrow
)


In [ ]:
def _extent(results):
    """Return [V_exit_min, V_exit_max, V_ent_min, V_ent_max] for imshow."""
    b = results['v_exit_grid']
    d = results['v_ent_grid']
    return [b[0], b[-1], d[0], d[-1]]


def _eta_clim(eta_arr, low_pct=2, hi_pct=99, floor=-5.0, ceil=1.5):
    """Auto color limits from eta array (skip NaN/Inf)."""
    v = eta_arr[np.isfinite(eta_arr)]
    if len(v) == 0:
        return -3.0, 1.0
    return (max(float(np.percentile(v, low_pct)), floor),
            min(float(np.percentile(v,  hi_pct)), ceil))


def plot_fig1c(results, st=None):
    """Fig.1(c): Variance heatmap at intermediate AL iteration + selected points."""
    if st is None: st = _ST
    snap = results.get('snapshot_data', {})
    if not snap:
        print("[SKIP] No snapshot data -- increase N_AL or change snapshot_round")
        return

    extent = _extent(results)
    fig, ax = plt.subplots(figsize=st['figsize_sq'])

    im = ax.imshow(snap['variance_map'], origin='lower', extent=extent,
                   aspect='auto', cmap=st['cmap_var'])
    plt.colorbar(im, ax=ax, label='k-NN variance')

    # Selected measurement points for next round (black dots, per paper)
    sel = snap['selected_points']
    ax.scatter(sel[:, 1], sel[:, 0], c='black', s=8, zorder=5,
               label=f"$N_\\mathrm{{meas}}={len(sel)}$ selected")

    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    if st['show_title']:
        ax.set_title(f"Fig.1(c): k-NN variance at AL round {snap['round']}\n"
                     f"(high variance = plateau boundaries, all n levels)")
    ax.legend(loc=st['legend_loc'], fontsize=st['fs_legend'])
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1c_variance.pdf'),
                bbox_inches='tight')
    plt.show()


def plot_fig1d(results, ef=None, eta_vmin=None, eta_vmax=None, st=None):
    """Fig.1(d): Sparse measured eta_M points after all AL iterations."""
    if st is None: st = _ST
    X = results['X_train']       # (N, 2) = (V_ent, V_exit)
    y = results['y_train']       # raw values
    extent = _extent(results)

    if ef is None:
        cfg = results['config']
        ef = cfg.e_charge * cfg.f_drive
    gain = results['gain']

    n_vals    = y * gain / ef
    deviation = np.abs(n_vals - 1.0)
    deviation = np.clip(deviation, 1e-15, None)    # avoid log(0)
    eta_vals  = np.log10(deviation)

    # Auto color limits -- data-driven
    if eta_vmin is None or eta_vmax is None:
        _lo, _hi = _eta_clim(eta_vals)
        if eta_vmin is None: eta_vmin = _lo
        if eta_vmax is None: eta_vmax = _hi

    fig, ax = plt.subplots(figsize=st['figsize_sq'])
    sc = ax.scatter(X[:, 1], X[:, 0], c=eta_vals, cmap=st['cmap_eta'],
                    s=4, vmin=eta_vmin, vmax=eta_vmax)
    plt.colorbar(sc, ax=ax,
                 label='$\\eta_M = \\log_{10}(|\\langle n \\rangle - 1|)$')
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    if st['show_title']:
        ax.set_title(f"Fig.1(d): Sparse measured $\\eta_M$ ({len(X)} points)\n"
                     f"color: [{eta_vmin:.2f}, {eta_vmax:.2f}]")
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1d_sparse_eta.pdf'),
                bbox_inches='tight')
    plt.show()


def plot_fig1e(results, ef=None, eta_noise=-3.5, eta_vmin=None, eta_vmax=None, st=None):
    """Fig.1(e): Interpolated eta_M map with eta_noise contour.

    NOTE: eta_M = log10(|<n>-1|) is n=1 specific.
    Regions with n=2, 3, ... appear as eta~0 (same as off state).
    Use plot_n_map() to visualise all plateau levels.
    """
    if st is None: st = _ST
    interp_map = results['interp_map']
    b = results['v_exit_grid']
    d = results['v_ent_grid']
    extent = _extent(results)

    if ef is None:
        cfg = results['config']
        ef = cfg.e_charge * cfg.f_drive
    gain = results['gain']

    # Convert interpolated map to <n> then eta
    n_map   = interp_map * gain / ef
    eta_map = compute_eta(n_map)

    # Auto color limits -- data-driven
    if eta_vmin is None or eta_vmax is None:
        _lo, _hi = _eta_clim(eta_map)
        if eta_vmin is None: eta_vmin = _lo
        if eta_vmax is None: eta_vmax = _hi

    fig, ax = plt.subplots(figsize=st['figsize_sq'])
    im = ax.imshow(eta_map, origin='lower', extent=extent,
                   aspect='auto', cmap=st['cmap_eta'],
                   vmin=eta_vmin, vmax=eta_vmax)
    plt.colorbar(im, ax=ax,
                 label='$\\eta_M = \\log_{10}(|\\langle n \\rangle - 1|)$')

    # eta_noise contour -- white dashed (inner noise-floor boundary)
    B, D = np.meshgrid(b, d)
    ax.contour(B, D, eta_map, levels=[eta_noise],
               colors='white', linestyles='--', linewidths=st['lw'])

    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    if st['show_title']:
        ax.set_title(f"Fig.1(e): Interpolated $\\eta_M$ map\n"
                     f"dashed = $\\eta_\\mathrm{{noise}}={eta_noise:.2f}$  "
                     f"| color: [{eta_vmin:.2f}, {eta_vmax:.2f}]\n"
                     f"(n=1 specific -- n>1 plateaus appear red; see <n> map)")
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1e_interp_eta.pdf'),
                bbox_inches='tight')
    plt.show()
    return eta_map


def plot_fig1f(results, eta_map, v_ent_idx=None,
               eta_noise=-3.5, eta_max=-0.6,
               x_margin=0.02, auto_xlim=True, st=None):
    """Fig.1(f): eta vs V_exit at fixed V_ent -- nonlinear Schoinas model fit.

    Visualization per lab-plot-style skill para 5:
    - Gray dots: all data points
    - Open red circles: fit points (eta < eta_max)
    - Red solid curve: Schoinas model eta(V)
    - Red dashed lines: left/right asymptotes
    - Dark-red star: eta_E^min marker at asymptote intersection
    - Gray dashed: eta_noise line
    - Text box: RMS residual

    auto_xlim : bool
        If True, zoom x-axis to the fit (plateau) region +/- x_margin.
    """
    if st is None: st = _ST
    b = results['v_exit_grid']
    d = results['v_ent_grid']

    # Auto-select V_ent row with the most negative eta (best plateau)
    if v_ent_idx is None:
        row_min_eta = np.nanmin(eta_map, axis=1)
        v_ent_idx   = int(np.nanargmin(row_min_eta))

    eta_line  = eta_map[v_ent_idx, :]
    v_ent_val = d[v_ent_idx]

    eta_E, info = extrapolate_eta_line(b, eta_line,
                                       eta_noise=eta_noise, eta_max=eta_max)

    fig, ax = plt.subplots(figsize=st['figsize_wide'])

    # All data points -- small gray dots
    ax.plot(b, eta_line, '.', color=st['c_data'], markersize=st['ms_data'],
            label='$\\eta_M$ data', zorder=2)

    if info:
        # Fit points -- open red circles
        ax.plot(b[info['fit_mask']], eta_line[info['fit_mask']],
                'o', mfc='none', mec=st['c_fit'],
                markersize=st['ms_fit'], linewidth=st['lw_asym'],
                label='fit points', zorder=3)

        # Schoinas model curve -- red solid
        ax.plot(info['V_model'], info['eta_model'],
                color=st['c_fit'], linewidth=st['lw_model'],
                label=f"Schoinas model (RMS = {info['rms']:.3f})", zorder=4)

        # Asymptotes -- red dashed
        v_asym = np.array([b[0], b[-1]])
        ax.plot(v_asym, info['a1'] * v_asym + info['b1'],
                color=st['c_fit'], linestyle='--', linewidth=st['lw_asym'],
                alpha=0.65,
                label=f"loading asymptote ($a_1={info['a1']:.1f}$)")
        ax.plot(v_asym, info['a2'] * v_asym + info['b2'],
                color=st['c_fit'], linestyle='--', linewidth=st['lw_asym'],
                alpha=0.65,
                label=f"emission asymptote ($a_2={info['a2']:.1f}$)")

        # eta_E^min -- dark-red star marker
        ax.plot(info['v_opt'], eta_E, '*', color=st['c_best'],
                markersize=st['ms_best'],
                zorder=5, label=f"$\\eta_E^{{\\min}} = {eta_E:.2f}$")
        ax.axhline(eta_E, color=st['c_best'], linestyle=':', linewidth=1,
                   alpha=0.5)

        # RMS residual text box
        ax.text(0.02, 0.05, f"RMS residual = {info['rms']:.3f}",
                transform=ax.transAxes, fontsize=st['fs_annot'],
                color=st['c_fit'],
                bbox=dict(boxstyle='round,pad=0.3',
                          facecolor='lightyellow', alpha=0.85))

    # eta_noise -- gray dashed
    ax.axhline(eta_noise, color=st['c_noise'], linestyle='--',
               linewidth=st['lw'],
               label=f"$\\eta_{{\\mathrm{{noise}}}} = {eta_noise:.2f}$")
    ax.axhline(eta_max, color='lightgray', linestyle='-.', linewidth=1,
               label=f"$\\eta_{{\\mathrm{{max}}}} = {eta_max}$ (fit cutoff)")

    # -- Auto y-scale from actual data range --------------------------
    eta_finite = eta_line[np.isfinite(eta_line)]
    if len(eta_finite) > 0:
        y_bot = max(float(np.nanmin(eta_finite)) - 0.5, -5.0)
        y_top = float(np.nanmax(eta_finite)) + 0.3
    else:
        y_bot, y_top = -4.0, 1.5
    ax.set_ylim(y_bot, y_top)

    # -- Auto x-scale: zoom to plateau region (eta < eta_max) + margin --
    if auto_xlim:
        plateau_mask = np.isfinite(eta_line) & (eta_line < eta_max)
        if plateau_mask.any():
            ax.set_xlim(float(b[plateau_mask].min()) - x_margin,
                        float(b[plateau_mask].max()) + x_margin)

    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$\\eta = \\log_{10}(|\\langle n \\rangle - 1|)$')
    if st['show_title']:
        ax.set_title(f"Fig.1(f): Schoinas $\\eta_E$ fit at "
                     f"$V_{{\\mathrm{{ent}}}} = {v_ent_val:.4f}$ V")
    ax.legend(fontsize=st['fs_legend'], loc=st['legend_loc'],
              ncol=st['legend_ncol'])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1f_eta_fit.pdf'),
                bbox_inches='tight')
    plt.show()
    return eta_E, info


def plot_fig1g(results, eta_map, eta_noise=-3.5, eta_max=-0.6, st=None):
    """Fig.1(g): 2D extrapolated eta_E(V_exit, V_ent) map."""
    if st is None: st = _ST
    b = results['v_exit_grid']
    d = results['v_ent_grid']
    extent = _extent(results)

    eta_E_map, eta_E_1d, fit_infos = extrapolate_eta_map(
        eta_map, b, d, eta_noise=eta_noise, eta_max=eta_max)

    # V_opt(V_ent) trajectory (Proposal A): 각 V_ent 행의 Schoinas 피팅
    # 결과 V_opt = (b2-b1)/(a1-a2). 오른쪽 패널의 η_E^min(V_ent)이
    # '어느 V_exit에서 읽힌 값인지'를 왼쪽 2D 맵 위에 시각화.
    V_opt_traj = np.array([info.get('v_opt', np.nan) if info else np.nan
                           for info in fit_infos])
    valid_traj = np.isfinite(V_opt_traj)

    # Auto color limits for eta_E map
    _vmin_g, _vmax_g = _eta_clim(eta_E_map)

    fig, axes = plt.subplots(1, 2, figsize=st['figsize_2col'])

    # Left: 2D eta_E map + V_opt(V_ent) trajectory overlay (Proposal A)
    ax = axes[0]
    im = ax.imshow(eta_E_map, origin='lower', extent=extent,
                   aspect='auto', cmap=st['cmap_eta'],
                   vmin=_vmin_g, vmax=_vmax_g)
    plt.colorbar(im, ax=ax,
                 label='$\\eta_E = \\log_{10}(|\\langle n \\rangle - 1|)$')
    if np.any(valid_traj):
        # Clip V_opt to V_exit plot range to avoid outliers distorting axis
        _vxmn, _vxmx = b.min(), b.max()
        _inr = valid_traj & (V_opt_traj >= _vxmn) & (V_opt_traj <= _vxmx)
        if np.any(_inr):
            ax.plot(V_opt_traj[_inr], d[_inr],
                    color='cyan', linewidth=st['lw'], alpha=0.9,
                    label='$V_\\mathrm{opt}(V_\\mathrm{ent})$')
            ax.legend(loc='upper right', fontsize=st['fs_legend'],
                      framealpha=0.85)
    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    if st['show_title']:
        ax.set_title('Fig.1(g): Extrapolated $\\eta_E(V_{exit}, V_{ent})$\n'
                     '(n=1 plateau envelope; cyan: $V_\\mathrm{opt}$ per row)')

    # Right: η_E^min vs V_ent (Proposal C: clarify self-optimal V_exit)
    ax = axes[1]
    valid = np.isfinite(eta_E_1d)
    if np.any(valid):
        ax.plot(d[valid], eta_E_1d[valid],
                color=st['c_conv'], marker='.', linestyle='-',
                markersize=st['ms_data'] + 1)
        best_idx   = int(np.nanargmin(eta_E_1d))
        best_V_opt = V_opt_traj[best_idx] if valid_traj[best_idx] else np.nan
        ax.axhline(eta_E_1d[best_idx], color=st['c_fit'],
                   linestyle='--', alpha=0.5)
        ax.annotate(f"best $\\eta_E^{{\\min}} = {eta_E_1d[best_idx]:.2f}$\n"
                    f"at $V_\\mathrm{{ent}} = {d[best_idx]:.4f}$ V\n"
                    f"($V_\\mathrm{{opt}} = {best_V_opt:.4f}$ V)",
                    xy=(d[best_idx], eta_E_1d[best_idx]),
                    xytext=st['annot_xy'], textcoords='axes fraction',
                    arrowprops=dict(arrowstyle='->', color=st['c_fit']),
                    fontsize=st['fs_annot'], color=st['c_fit'])
    ax.set_xlabel('$V_{ent}$ (V)')
    ax.set_ylabel('$\\eta_E^{\\min}$ (at self-optimal $V_\\mathrm{exit}$)')
    if st['show_title']:
        ax.set_title('$\\eta_E^{\\min}$ vs $V_\\mathrm{ent}$\n'
                     '(each point: $V_\\mathrm{exit}=V_\\mathrm{opt}(V_\\mathrm{ent})$)')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'fig1g_eta_E_map.pdf'),
                bbox_inches='tight')
    plt.show()
    return eta_E_map, eta_E_1d


def plot_n_map(results, ef=None, n_contour_levels=None, st=None):
    """<n> heatmap -- shows ALL quantization plateaus (n=1, 2, 3, ...).

    This complements eta_M map (Fig.1e), which is n=1 specific.
    Contour lines at half-integer values mark plateau boundaries.
    """
    if st is None: st = _ST
    interp_map = results['interp_map']
    gain = results['gain']
    if ef is None:
        cfg = results['config']
        ef = cfg.e_charge * cfg.f_drive
    n_map = interp_map * gain / ef

    b = results['v_exit_grid']
    d = results['v_ent_grid']
    extent = _extent(results)

    n_min_d = float(np.nanmin(n_map))
    n_max_d = float(np.nanmax(n_map))

    # Plateau boundary contours at 0.5, 1.5, 2.5, ...
    if n_contour_levels is None:
        k_lo = max(0, int(np.floor(n_min_d)))
        k_hi = int(np.ceil(n_max_d))
        n_contour_levels = [k + 0.5 for k in range(k_lo, k_hi)]

    fig, ax = plt.subplots(figsize=st['figsize_sq'])
    im = ax.imshow(n_map, origin='lower', extent=extent,
                   aspect='auto', cmap=st['cmap_n'],
                   vmin=max(n_min_d, -0.5), vmax=min(n_max_d + 0.5, 6.0))
    cb = plt.colorbar(im, ax=ax, label='$\\langle n \\rangle = I / (ef)$')

    if n_contour_levels:
        B, D = np.meshgrid(b, d)
        cs = ax.contour(B, D, n_map, levels=n_contour_levels,
                        colors='white', linestyles='--',
                        linewidths=st['lw'])
        ax.clabel(cs,
                  fmt={v: f'n={v+0.5:.0f}' for v in n_contour_levels},
                  fontsize=st['fs_legend'], inline=True)

    ax.set_xlabel('$V_{exit}$ (V)')
    ax.set_ylabel('$V_{ent}$ (V)')
    if st['show_title']:
        ax.set_title(f'$\\langle n \\rangle$ map -- all plateau levels\n'
                     f'dashed lines = plateau boundaries (n = 0.5, 1.5, ...)\n'
                     f'$\\langle n \\rangle$ range: [{n_min_d:.2f}, {n_max_d:.2f}]')
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'n_map.pdf'),
                bbox_inches='tight')
    plt.show()
    return n_map


def plot_al_convergence(results, st=None):
    """AL convergence: max variance vs round (supplementary)."""
    if st is None: st = _ST
    history = results['al_history']
    if not history:
        print("[SKIP] No AL history")
        return

    history = np.array(history)
    fig, ax = plt.subplots(figsize=st['figsize_conv'])
    ax.semilogy(history[:, 0], history[:, 2],
                color=st['c_conv'], marker='.', linestyle='-')
    ax.set_xlabel('AL Round')
    ax.set_ylabel('Max k-NN variance')
    if st['show_title']:
        ax.set_title('AL Convergence')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'al_convergence.pdf'),
                bbox_inches='tight')
    plt.show()


def plot_voltage_steps(results, st=None):
    """Voltage step size histogram (cf. paper Fig.2 inset)."""
    if st is None: st = _ST
    X = results['X_train']
    diffs = np.diff(X, axis=0)
    # X columns: (V_ent, V_exit)
    step_sizes = np.linalg.norm(diffs, axis=1) * 1000   # mV

    fig, axes = plt.subplots(1, 2, figsize=st['figsize_hist'])

    ax = axes[0]
    ax.plot(step_sizes, color=st['c_conv'], alpha=0.5, linewidth=0.5)
    ax.set_xlabel('Measurement index')
    ax.set_ylabel('Step size (mV)')
    if st['show_title']:
        ax.set_title('Voltage step sizes')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.hist(step_sizes, bins=50, color=st['c_step'], edgecolor='black')
    ax.set_xlabel('Step size (mV)')
    ax.set_ylabel('Count')
    if st['show_title']:
        ax.set_title(f"mean={np.mean(step_sizes):.1f} mV, "
                     f"max={np.max(step_sizes):.1f} mV")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(results['output_dir'], 'voltage_steps.pdf'),
                bbox_inches='tight')
    plt.show()


## Run: Hardware Mode

Uncomment and configure for real pump measurements.  
**CRITICAL**: Verify GPIB addresses and Ithaco gain before running.

In [ ]:
# -- Hardware Run -----------------------------------------------------------
# v3: 모든 실험 파라미터는 cell[3] ALSMConfig에서 편집.
# 이 셀은 인스턴스 생성·연결·실행·안전 종료만 담당 (override 없음).
#
# connect()를 try 블록 안에서 호출해야 연결 도중 실패하더라도 close()가
# 실행되어 안전 종료(ramp_to_safe)가 보장됨. close()는 OD 재쿼리로 추적
# 변수를 최신화한 뒤 램프하므로 부분 연결 상태에서도 안전함.

cfg_hw  = ALSMConfig()
ctrl_hw = InstrumentController(cfg_hw)
try:
    ctrl_hw.connect()
    results_hw = driver_alsm(cfg_hw, ctrl_hw)

    # eta_noise: 실측 데이터에서 추정 (skill §4 — 보간 데이터 사용 금지)
    _ef_hw = cfg_hw.e_charge * cfg_hw.f_drive
    eta_noise_hw = find_eta_noise(results_hw['y_train'], results_hw['gain'], _ef_hw)
    print(f'eta_noise (10th percentile): {eta_noise_hw:.3f}')

    plot_fig1c(results_hw)
    plot_fig1d(results_hw)
    eta_map_hw = plot_fig1e(results_hw, eta_noise=eta_noise_hw)
    plot_fig1f(results_hw, eta_map_hw, eta_noise=eta_noise_hw, eta_max=cfg_hw.eta_max)
    plot_fig1g(results_hw, eta_map_hw, eta_noise=eta_noise_hw, eta_max=cfg_hw.eta_max)
    plot_al_convergence(results_hw)
    plot_voltage_steps(results_hw)
finally:
    ctrl_hw.close()    # 항상 안전 종료 (skill §4)


In [ ]:
# ── Hardware Mode: Fig.1 Visualization ──
# Run this cell after the hardware run above completes (results_hw must exist).

_ef_hw = cfg_hw.e_charge * cfg_hw.f_drive

# Estimate η_noise from actual measured data (skill §4 — 10th percentile)
eta_noise_hw = find_eta_noise(results_hw['y_train'], results_hw['gain'], _ef_hw)
eta_max_hw   = cfg_hw.eta_max
print(f"η_noise (10th percentile of {len(results_hw['y_train'])} measurements): "
      f"{eta_noise_hw:.3f}")

# (c) Variance heatmap at AL round 3
plot_fig1c(results_hw)

# (d) Sparse measured η_M
plot_fig1d(results_hw)

# (e) Interpolated η_M map
eta_map_hw = plot_fig1e(results_hw, eta_noise=eta_noise_hw)

# ⟨n⟩ map — all plateau levels (n=1 η_M map only shows n=1; this shows everything)
plot_n_map(results_hw)

# (f) η vs V_exit — Schoinas nonlinear fit (skill §2), auto-zoomed to n=1 plateau
plot_fig1f(results_hw, eta_map_hw, eta_noise=eta_noise_hw, eta_max=eta_max_hw)

# (g) 2D extrapolated η_E map
eta_E_map_hw, eta_E_1d_hw = plot_fig1g(
    results_hw, eta_map_hw, eta_noise=eta_noise_hw, eta_max=eta_max_hw)

# Supplementary
plot_al_convergence(results_hw)
plot_voltage_steps(results_hw)

## Post-processing: Load & Replot from Saved Output

Reproduce all figures from a previous hardware run without re-measuring.

| Function | Input | Notes |
|----------|-------|-------|
| `load_results_from_dir(output_dir)` | `output_ALSM_*/` | Loads `results.npz` + parses `accum.txt` header. **Preferred**. |
| `load_results_from_accum(accum_path)` | `accum.txt` path | Re-interpolates from raw data only. |
| `replot_all(results)` | results dict | Calls all plot functions. |

In [ ]:
def _parse_accum_header(accum_path):
    """Parse metadata from accum.txt header comments."""
    gain = 1e-9
    ef   = None
    with open(accum_path) as f:
        for line in f:
            if not line.startswith('#'):
                break
            if 'gain=' in line:
                try:
                    gain = float(line.split('gain=')[1].split()[0])
                except Exception:
                    pass
            if 'ef=' in line:
                try:
                    ef = float(line.split('ef=')[1].split()[0])
                except Exception:
                    pass
    return gain, ef


def _make_cfg(ef):
    """Build a minimal config object from ef value."""
    class _Cfg:
        eta_max  = -0.6
        I_sat_nA = 5.0
    cfg = _Cfg()
    cfg.e_charge = 1.602176634e-19
    cfg.f_drive  = ef / cfg.e_charge if ef is not None else 1.0
    if ef is None:
        cfg.e_charge = 1.0
    return cfg


def load_results_from_dir(output_dir):
    """Load all results from a saved output directory (results.npz + accum.txt)."""
    npz_path   = os.path.join(output_dir, 'results.npz')
    accum_path = os.path.join(output_dir, 'accum.txt')

    gain, ef = _parse_accum_header(accum_path)
    cfg      = _make_cfg(ef)

    d = np.load(npz_path, allow_pickle=False)
    al_hist = (d['al_history'].tolist()
               if d['al_history'].ndim > 0 and len(d['al_history']) > 0 else [])

    results = {
        'interp_map':    d['interp_map'],
        'X_train':       d['X_train'],
        'y_train':       d['y_train'],
        'v_exit_grid':   d['v_exit_grid'],
        'v_ent_grid':    d['v_ent_grid'],
        'measured_mask': d['measured_mask'],
        'al_history':    al_hist,
        'snapshot_data': {},
        'output_dir':    output_dir,
        'total_time':    0.0,
        'config':        cfg,
        'gain':          float(d['gain']),
    }
    print(f"[LOAD] {output_dir}")
    print(f"  X_train : {d['X_train'].shape[0]} points")
    print(f"  grid    : {d['interp_map'].shape}")
    print(f"  gain    : {float(d['gain']):.0e} A/V")
    if ef is not None:
        print(f"  ef      : {ef:.4e} A  ({ef*1e12:.3f} pA)")
    return results


def load_results_from_accum(accum_path, n_exit=200, n_ent=200, output_dir=None):
    """Reconstruct results from accum.txt alone (re-interpolates raw data)."""
    if output_dir is None:
        output_dir = os.path.dirname(os.path.abspath(accum_path))

    gain, ef = _parse_accum_header(accum_path)
    cfg      = _make_cfg(ef)

    data       = np.loadtxt(accum_path)
    V_ent_all  = data[:, 0]
    V_exit_all = data[:, 1]
    y_raw_all  = data[:, 2]

    b      = np.linspace(V_exit_all.min(), V_exit_all.max(), n_exit)
    d_grid = np.linspace(V_ent_all.min(),  V_ent_all.max(),  n_ent)
    B, D   = np.meshgrid(b, d_grid)
    grid_pts = np.column_stack([D.ravel(), B.ravel()])
    X_pts    = np.column_stack([V_ent_all, V_exit_all])

    interp_vals = griddata(X_pts, y_raw_all, grid_pts, method='linear')
    nan_mask = np.isnan(interp_vals)
    if nan_mask.any():
        near_vals = griddata(X_pts, y_raw_all, grid_pts[nan_mask], method='nearest')
        interp_vals[nan_mask] = near_vals
    interp_map = interp_vals.reshape(n_ent, n_exit)

    results = {
        'interp_map':    interp_map,
        'X_train':       X_pts,
        'y_train':       y_raw_all,
        'v_exit_grid':   b,
        'v_ent_grid':    d_grid,
        'measured_mask': np.zeros((n_ent, n_exit), dtype=bool),
        'al_history':    [],
        'snapshot_data': {},
        'output_dir':    output_dir,
        'total_time':    0.0,
        'config':        cfg,
        'gain':          gain,
    }
    print(f"[LOAD-ACCUM] {os.path.basename(accum_path)}")
    print(f"  {len(data)} pts | gain={gain:.0e} A/V"
          + (f" | ef={ef:.4e} A" if ef is not None else ""))
    return results


def replot_all(results, output_dir=None):
    """Reproduce all Fig.1 plots from a loaded results dict."""
    if output_dir is not None:
        results = dict(results)
        results['output_dir'] = output_dir
        os.makedirs(output_dir, exist_ok=True)

    cfg = results['config']
    ef  = cfg.e_charge * cfg.f_drive
    eta_noise = find_eta_noise(results['y_train'], results['gain'], ef)
    eta_max   = cfg.eta_max
    print(f"η_noise = {eta_noise:.3f}  (10th pct of {len(results['y_train'])} pts)")

    plot_fig1c(results)
    plot_fig1d(results)
    eta_map = plot_fig1e(results, eta_noise=eta_noise)
    plot_n_map(results)
    plot_fig1f(results, eta_map, eta_noise=eta_noise, eta_max=eta_max)
    eta_E_map, eta_E_1d = plot_fig1g(results, eta_map,
                                      eta_noise=eta_noise, eta_max=eta_max)
    plot_al_convergence(results)
    plot_voltage_steps(results)
    return eta_map, eta_E_map, eta_E_1d

In [ ]:
# ── Reload & replot from a saved hardware run directory ──
#
# Option A: preferred (uses interpolated map from results.npz)
# OUTPUT_DIR = "output_ALSM_20260418_123456"   # ← your output directory
# results_loaded = load_results_from_dir(OUTPUT_DIR)
# eta_map_l, eta_E_map_l, eta_E_1d_l = replot_all(results_loaded)
#
# Option B: re-interpolate from accum.txt only
# results_loaded = load_results_from_accum(
#     os.path.join(OUTPUT_DIR, "accum.txt"),
#     n_exit=200, n_ent=200
# )
# replot_all(results_loaded)
#
# Option C: save figures to a new directory
# replot_all(results_loaded, output_dir="replot_output")